In [2]:
%%writefile requirements.txt
numpy==2.3.3
pandas==2.3.3
pmdarima==2.1.1
polars==1.34.0
polars-runtime-32==1.34.0
scikit-learn==1.7.2
statsmodels==0.14.5
xgboost==3.1.1
yfinance==0.2.66

Overwriting requirements.txt


In [3]:
import os

def make_directory(path: str) -> None:
    try:
        os.makedirs(path)
        print(f"created directory '{path}'!")
    except FileExistsError:
        print(f"directory '{path}' already exists!")

make_directory("src")
make_directory("workflows")
make_directory("src/xgboost_model")
make_directory("src/sarimax_model")
make_directory("src/prophet_model")

directory 'src' already exists!
directory 'workflows' already exists!
directory 'src/xgboost_model' already exists!
directory 'src/sarimax_model' already exists!
directory 'src/prophet_model' already exists!


# loading the raw data from `yfinance`

this serves as the base function for loading all data

In [4]:
%%writefile src/load_data.py

"""
function written to easily load and process stock data from `yfinance`.
"""

import pandas as pd
import yfinance as yf
import polars as pl

def load_stocks(stocks: list, start: str, end: str, use_polars: bool = True):
    """load stock data from `yfinance`.

    stocks are loaded singularly, from start to end date. option to return a 
    polars dataframe or a pandas dataframe.

    Args:
        stocks (list): single-item list of stock tockers.
        start (str): first historical date.
        end (str): last historical date (up to today).
        use_polars (bool, optional): whether to return a polars dataframe or a
            pandas dataframe. defaults to true.

    Raises:
        ValueError: raised if users enter more than 1 ticker

    Returns:
        DataFrame: polars or pandas dataframe, depending on the value of
        `use_polars`.
    """
    if len(stocks) > 1:
        raise ValueError("can only do one stock forecast at a time")
    df = yf.download(stocks, start, end)
    df.index = pd.to_datetime(df.index)
    df.columns = (
        pd.MultiIndex.from_tuples(df.columns) 
        if not isinstance(df.columns, pd.MultiIndex) else df.columns
    )
    df.columns = df.columns.set_names(["Field", "Ticker"])
    df.index.name = "Date"
    df = df.apply(pd.to_numeric, errors="coerce")

    df_out = (
        df.swaplevel("Field", "Ticker", axis=1)
        .sort_index(axis=1)
        .stack("Ticker", future_stack=True)
        .reset_index()
    )

    df_out = df_out.rename(columns=str.lower)

    return pl.from_pandas(df_out) if use_polars else df_out

Overwriting src/load_data.py


# data transformations and feature engineering

## xgboost

In [5]:
%%writefile src/xgboost_model/xgboost_etl.py

"""
set of functions to process `yfinance` data for the XGBoost model.

adds lagged features and time indicators to build the full feature space for
each model.
"""

import polars as pl
from datetime import datetime, date, timedelta
import yfinance as yf
from typing import Tuple

from src.load_data import load_stocks


def prep_columns(df: pl.DataFrame, col: str) -> pl.DataFrame:
    """prep the columns pulled from `yfinance` into a clean dataframe

    adds a 'move' column to indicate overall daily change; time-lapse columns
    lagged over 1, 7, 30 days; and rolling mean and SD values over 7 days.

    Args:
        df (pl.DataFrame): raw `yfinance` dataframe.
        col (str): which column (choose between 'close', 'open') to compute the
            lag features for.

    Returns:
        pl.DataFrame: full dataframe with lagged features of chosen column.
    """
    if col == "move":
        df = df.with_columns(
            (pl.col("close") - pl.col("open")).alias(col)
        )

    df_out =  (
        df.select(
            [
                "date",
                "ticker",
                "volume",
                col
            ]
        )
        .sort([pl.col("ticker"), pl.col("date")], descending=False)
        .with_columns(
            pl.col(col).shift(1).over("ticker").alias(f"prev1_{col}"),
            pl.col(col).shift(7).over("ticker").alias(f"prev7_{col}"),
            pl.col(col).shift(30).over("ticker").alias(f"prev30_{col}"),
        )
        .with_columns(
            pl.col(col)
            .rolling_mean(window_size=7, min_samples=2)
            .shift(1)
            .over("ticker")
            .alias(f"{col}_rolling_mean_7")
        )
        .with_columns(
            pl.col(col)
            .rolling_std(window_size=7, min_samples=2)
            .shift(1)
            .over("ticker")
            .alias(f"{col}_rolling_std_7")
        )
    )

    return df_out


def prep_data_frame(df: pl.DataFrame) -> pl.DataFrame:
    """prepare lag columns for all of 'open', 'close', and 'move'.

    Args:
        df (pl.DataFrame): raw `yfinance` stock dataframe.

    Returns:
        pl.DataFrame: processed stock data with lag columns for all price
            indicators.
    """
    markers = ["open", "close", "move"]
    df_out = None

    for marker in markers:
        df_prep = prep_columns(df, marker)
        if df_out is None:
            df_out = df_prep
        else:
            df_out = df_prep.join(
                df_out, on=["date", "ticker", "volume"], how="inner"
            )
    
    return (
        df_out.with_columns(
            pl.col("date").dt.weekday().alias("dow")
        )
        .with_columns(
            pl.col("date").dt.month().alias("month")
        )
        .with_columns(
            pl.when(pl.col("dow").is_in([0, 4]))
            .then(pl.lit(1))
            .otherwise(pl.lit(0))
            .alias("mon_or_fri")
        )
    )

def build_dataset(df: pl.DataFrame, label: str = "close") -> pl.DataFrame:
    """wrapper for `prep_data_frame`.

    Args:
        df (pl.DataFrame): raw `yfinance` stock dataframe.
        label (str, optional): which of 'close' or 'move' to process. Defaults
            to "close".

    Raises:
        ValueError: only accepts 'close' or 'move'.

    Returns:
        pl.DataFrame: fully processed `yfinance` data with nulls removed.
    """
    df_feat = prep_data_frame(df)

    if label == "close":
        df_feat = df_feat.with_columns(pl.col("close").shift(-1).alias("label"))
    elif label == "move":
        df_feat = df_feat.with_columns(pl.col("move").shift(-1).alias("label"))
    else:
        raise ValueError("label must be one of ['close', 'move']")
    
    return df_feat.drop_nulls()

Overwriting src/xgboost_model/xgboost_etl.py


## sarimax

In [6]:
%%writefile src/sarimax_model/sarimax_etl.py

"""
set of functions to process `yfinance` data for the SARIMAX model.

pulls code from xgboost model (`build_dataset`) and uses it to add indexes to 
the dataframe. 
"""

import polars as pl
from datetime import datetime, date, timedelta
import yfinance as yf
from typing import Tuple

from src.load_data import load_stocks
from src.xgboost_model.xgboost_etl import build_dataset


def build_df_with_indices(
    df: pl.DataFrame, label: str, start: str, end: str
) -> pl.DataFrame:
    """process raw dataframe and add indices.

    this is originally intended to bolster the SARIMAX model by adding broad
    exogenous variables to the features space.

    Args:
        df (pl.DataFrame): raw `yfiance` stock dataframe.
        label (str): 'close' or 'move' price indicator.
        start (str): start date for index pulling from `yfinance`.
        end (str): end date for index pulling from `yfinance`.

    Returns:
        pl.DataFrame: features data with lagged columns and added index values.
    """
    indices_list = ["SPY", "QQQ", "IWM", "VXX", "UUP", "HYG", "LQD"]
    df_idx_out = None

    for idx in indices_list:
        idx_cl = idx.replace("^", "")

        df_idx = load_stocks([idx], start, end).select(
            pl.col("date"),
            pl.col("close").alias(f"{idx_cl}_close"),
            pl.col("volume").alias(f"{idx_cl}_volume")
        )

        if df_idx_out is None:
            df_idx_out = df_idx
        else:
            df_idx_out = df_idx_out.join(df_idx, on=["date"], how="inner")
    
    df_ticker = build_dataset(df, label)

    df_out = df_ticker.join(df_idx_out, on=["date"], how="inner")
    
    return df_out

Overwriting src/sarimax_model/sarimax_etl.py


## prophet

In [7]:
%%writefile src/prophet_model/prophet_etl.py

"""
set of functions to process `yfinance` data for the Prophet model.

focuses on pulling indexes and computing returns and volatility to add these as
regressors in Prophet.
"""

import yfinance as yf 
import polars as pl 
import pandas as pd
from typing import Tuple

from src.load_data import load_stocks 


class GetSectorETF:
    """simple class to infer sector ETFs"""
    def __init__(
        self,
        indexes: list,
        label: str,
        target_ticker: str
    ):
        self.SECTOR_TO_ETF = {
            "Technology": "XLK",
            "Communication Services": "XLC",
            "Financial Services": "XLF",
            "Energy": "XLE",
            "Consumer Cyclical": "XLY",
            "Consumer Defensive": "XLP",
            "Industrials": "XLI",
            "Healthcare": "XLV",
            "Real Estate": "XLRE",
            "Utilities": "XLU",
        }

        self.indexes = indexes
        self.target_ticker = target_ticker

        if label not in ["open", "close", "move"]:
            raise ValueError("label must be one of ['open', 'close', 'move]")
        else:
            self.label = label

    def extract_stock_info(self, stock_df: pl.DataFrame) -> Tuple[str, str, str]:
        """extract the ticker, earliest, and latest data from `stock_df`.

        Args:
            stock_df (pl.DataFrame): stock dataframe loaded from `yfinance`.

        Returns:
            Tuple[str, str, str]: ticker, start_date, end_date values
        """
        tickers = [stock_df.select("ticker").unique().item()]

        date_df = (
            stock_df
            .select("date")
            .unique()
            .sort(by="date", descending=True)
        )

        min_date = date_df.select("date").tail(1).item().strftime("%Y-%m-%d")
        max_date = date_df.select("date").head(1).item().strftime("%Y-%m-%d")

        return tickers, min_date, max_date

    def infer_sector_etfs(self, stock_df: pl.DataFrame) -> pl.DataFrame:
        """using tickers, extracts the close values of sector ETFs.

        Args:
            stock_df (pl.DataFrame): stock dataframe loaded from `yfinance`.

        Returns:
            pl.DataFrame: dataframe of dates, `label` values for relevant ETFs 
                for the stock's ticker, as well as `label` and values for the
                passed-in indexes.
        """
        tickers, start_date, end_date = self.extract_stock_info(stock_df)

        etfs = set()

        for t in tickers:
            info = yf.Ticker(t).info
            sector = info.get("sector")
            if sector in self.SECTOR_TO_ETF:
                etfs.add(self.SECTOR_TO_ETF[sector])
        
        ticker_list = list(etfs)
        ticker_list += self.indexes
        df_out = None

        for ticker in ticker_list:
            df_temp = load_stocks([ticker], start_date, end_date).select(
                pl.col("date"),
                pl.col(self.label).alias(f"{ticker}")
            )

            if df_out is None:
                df_out = df_temp 
            else:
                df_out = df_out.join(df_temp, on=["date"], how="inner")
        
        return df_out
    
    def build_etf_df(self, stock_df: pl.DataFrame) -> pl.DataFrame:
        """combine ETF values with stock dataframe

        Args:
            stock_df (pl.DataFrame): stock dataframe loaded from `yfinance`.

        Returns:
            pl.DataFrame: combined dataframe of `stock_df` with ETF values.
        """
        df_etf = self.infer_sector_etfs(stock_df)

        return (
            stock_df.select(
                pl.col("date"), pl.col(self.label).alias(self.target_ticker)
            )
            .join(
                df_etf, on=["date"], how="inner"
            )
        )

def compute_returns(df: pl.DataFrame) -> pl.DataFrame:
    """compute the daily return rate.

    Args:
        df (pl.DataFrame): stock dataframe loaded from `yfinance` and processed
            through `GetSectorETF`.

    Returns:
        pl.DataFrame: polars dataframe with daily returns for a target stock and
            desired indexes.
    """
    tickers = [c for c in df.columns if c != "date"]

    for ticker in tickers:
        df = df.with_columns(
            pl.col(ticker).pct_change().alias(f"{ticker}_return")
        )
    
    return df

def compute_return_volatility(df: pl.DataFrame, window: int) -> pd.DataFrame:
    """compute the rolling volatility (SD) of target stock and indexes.

    returns a pandas dataframe for use in prepping data for Prophet.

    Args:
        df (pl.DataFrame): polars dataframe with target stock and index prices
            and returns.
        window (int): rolling window (in days) to compute volatility.

    Returns:
        pd.DataFrame: pandas dataframe with prices, returns, and return 
            volatility.
    """
    tickers = [c for c in df.columns if c.endswith("_return")]

    for ticker in tickers:
        df = df.with_columns(
            pl.col(ticker)
            .rolling_std(window_size=window, min_samples=1)
            .alias(f"{ticker}_return_rolling_std_{window}")
        )
    
    return df.to_pandas()

def compute_rsi(df: pd.DataFrame, ticker: str, period: int) -> pd.Series:
    """add a column for RSI over a period window.

    Args:
        df (pd.DataFrame): pandas dataframe with target stock PRICES.
        ticker (str): stock to compute RSI for.
        period (int): window (days).

    Returns:
        pd.Series: pandas series to add to `df` as a column containing RSI
            values.
    """
    prices = pd.to_numeric(df[ticker], errors="coerce")
    delta = prices.diff()

    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)

    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return rsi

def compute_price_volatility(
    df: pd.DataFrame, ticker: str, period: int
) -> pd.Series:
    """add price volatility columns.

    Args:
        df (pd.DataFrame): pandas dataframe with price data.
        ticker (str): stock to compute volatility for.
        period (int): window (days).

    Returns:
        pd.Series: column indicating lagged prices.
    """
    return df[ticker].shift(period)

def build_prophet_df(
    stocks: list,
    start: str,
    end: str,
    label: str,
    indexes: list
) -> pd.DataFrame:
    """run through suite of functions to build the final dataset for Prophet.

    Args:
        stocks (list): _description_
        start (str): _description_
        end (str): _description_
        label (str): _description_
        indexes (list): _description_

    Returns:
        pd.DataFrame: _description_
    """
    tkr = stocks[0]

    df_raw = load_stocks(stocks, start, end)
    etfs = GetSectorETF(indexes=indexes, label=label, target_ticker=tkr)
    df_etf = etfs.build_etf_df(df_raw)

    df_ret = compute_returns(df_etf)
    df_vol = compute_return_volatility(df_ret, 10)

    df_vol["rsi_7"] = compute_rsi(df_vol, tkr, 7)
    df_vol["rsi_14"] = compute_rsi(df_vol, tkr, 14)
    df_vol["rsi_21"] = compute_rsi(df_vol, tkr, 21)
    df_vol["prev7_close"] = compute_price_volatility(df_vol, tkr, 1)
    df_vol["prev14_close"] = compute_price_volatility(df_vol, tkr, 7)
    df_vol["prev30_close"] = compute_price_volatility(df_vol, tkr, 30)

    return df_vol.rename(
        columns={"date": "ds", f"{tkr}": "y"}
    ).dropna()

Overwriting src/prophet_model/prophet_etl.py


# prep the data for modeling

mostly just used for train-test splits but different models call for different  
data parsing (even if only slightly)

In [8]:
%%writefile src/model_preprocess.py

""" 
handle the creation of train-test splits for each model. not all are the same.
split for xgboost is traditional (X,y train/test tables), but for forecasting
models the split is a train/eval split without a test dataframe.

each split is done based on a cutoff date to only allow training on past data 
and testing/eval on future data.
"""

from datetime import datetime
import polars as pl
import pandas as pd
from typing import Tuple

def train_test_split_cutoff(
    df: pl.DataFrame, cutoff: datetime, label: str
) -> Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """do a train-test split based on a cutoff value.

    training data is all data prior to the cutoff, testing data is all data on
    or after the cutoff.

    Args:
        df (pl.DataFrame): dataframe to do the split on.
        cutoff (datetime): datetime object indicating the split date.
        label (str): label column to indicate which is 'y'.

    Returns:
        Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame]: gives the
            "traditional" X_train, X_test, y_train, y_test output (akin to
            sklearn).
    """
    train = df.filter(pl.col("date") < cutoff)
    test = df.filter(pl.col("date") >= cutoff)

    X_train, X_test = (
        train.drop(label, "ticker", "date"),
        test.drop(label, "ticker", "date")
    )
    y_train, y_test = train[label], test[label]

    return X_train, X_test, y_train, y_test

def split_ar_on_cutoff(
    df: pl.DataFrame, cutoff: datetime, label: str, exog_feats: list
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """split autoregressive data on a cutoff value.

    this doesn't return unique tables for X and y and is specifically designed
    for SARIMAX. can also be used for training Prophet. a pandas dataframe is
    returned (not polars) for use in the forecasting models.

    Args:
        df (pl.DataFrame): dataframe to do the split on.
        cutoff (datetime): datetime object indicating the split date.
        label (str): label column to indicate which is the endogenous value.
        exog_feats (list): list of exogenous features for SARIMAX.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: train/eval dataframes returned as
            pandas dataframes.
    """
    train = df.filter(pl.col("date") < cutoff)
    eval = df.filter(pl.col("date") >= cutoff)

    train_pd = train.to_pandas()
    eval_pd = eval.to_pandas()

    chg_cols = [f"{label}_rolling_std_7"]

    cols_list = [label] + exog_feats + chg_cols

    train_pd = train_pd[cols_list]
    eval_pd = eval_pd[cols_list]

    return train_pd, eval_pd

def split_prophet_df(
        df: pd.DataFrame, cutoff: datetime
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """train/eval split for Prophet model.

    Args:
        df (pd.DataFrame): pandas dataframe set up for Prophet.
        cutoff (datetime): datetime object indicating the split date.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: train/eval dataframes returned as
            pandas dataframes.
    """
    df_train = df[df["ds"] < cutoff]
    df_eval = df[df["ds"] >= cutoff]

    return df_train, df_eval

Overwriting src/model_preprocess.py


leave this here for now  
trying to mess with `argparse` so it can run in the command line but i'll save  
that for last....

In [9]:
# import argparse

# ### parameters (use argparse module)

# # model hyperparameters
# DEFAULT_NUM_ESTIMATORS = 100
# DEFAULT_LEARNING_RATE = 0.01

# parser = argparse.ArgumentParser(
#     description="model hyperparameters"
# )

# parser.add_argument(
#     "-NUM_ESTIMATORS",
#     type=int,
#     default=DEFAULT_NUM_ESTIMATORS,
#     help="number of estimators"
# )
# parser.add_argument(
#     "-LEARNING_RATE",
#     type=float,
#     default=DEFAULT_LEARNING_RATE,
#     help="how fast the model learns"
# )

# # data parameters
# DEFAULT_LABEL = "move"

# parser.add_argument(
#     "-START_DATE",
#     type=str,
#     default=None,
#     help="data training start date"
# )

# parser.add_argument(
#     "-END_DATE",
#     type=str,
#     default=None,
#     help="data training end date"
# )

# parser.add_argument(
#     "-STOCKS",
#     type=list,
#     default=None,
#     help="stocks to forecast"
# )

# parser.add_argument(
#     "-LABEL",
#     type=str,
#     default=DEFAULT_LABEL,
#     help="one of 'move', 'open', 'close'; which of these values to forecast"
# )

# # cutoff value
# parser.add_argument(
#     "-CUTOFF",
#     type=datetime,
#     default=None,
#     help="cutoff value for train/test split"
# )

# # create args
# args = parser.parse_args()

# NUM_ESTIMATORS = args.NUM_ESTIMATORS
# LEARNING_RATE = args.LEARNING_RATE
# STOCKS = args.STOCKS
# START_DATE = args.START_DATE
# END_DATE = args.END_DATE
# LABEL = args.LABEL
# CUTOFF = args.CUTOFF

# NOW build the model

# training the model

## xgboost

In [10]:
%%writefile src/xgboost_model/train_xgboost_model.py

"""
contains a wrapper function for loading the data and training the XGBoost model.
forecasting is not done here, only model training.
"""

import polars as pl
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error
from datetime import datetime
from typing import Tuple

from src.load_data import load_stocks
from src.xgboost_model.xgboost_etl import *
from src.model_preprocess import train_test_split_cutoff


def train_xgb_model(
    stocks: list,
    start_date: str,
    end_date: str,
    cutoff: datetime,
    label: str,
    n_estimators: int,
    learning_rate: float,
) -> Tuple[xgb.XGBRegressor, float, pl.DataFrame, list]:
    """load raw data, preprocess, and train XGBoost model.

    Args:
        stocks (list): single-item list of stock tickers.
        start_date (str): when to start the dataframe.
        end_date (str): final date of the dataframe.
        cutoff (datetime): cutoff datetime object for train/test splits.
        label (str): label column (y).
        n_estimators (int): XGBoost `n_estimators` hyperparameter.
        learning_rate (float): XGBoost `learning_rate` hyperparameter.

    Raises:
        ValueError: cannot process more than one stock at a time.

    Returns:
        Tuple[xgb.XGBRegressor, float, pl.DataFrame, list]: XGBoost regression
            model, RMSE value, full dataframe with features, features list.
    """
    if len(stocks) > 1:
        raise ValueError("can only do one stock forecast at a time")

    df_raw = load_stocks(
        stocks=stocks,
        start=start_date,
        end=end_date,
        use_polars=True
    )

    df_feat = build_dataset(df=df_raw, label=label)

    X_train, X_test, y_train, y_test = train_test_split_cutoff(
        df=df_feat, cutoff=cutoff, label="label"
    )

    model = xgb.XGBRegressor(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.005,
        objective="reg:squarederror"
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    rmse = root_mean_squared_error(y_test, preds)

    feature_cols = X_train.to_pandas().columns.tolist()

    return model, rmse, df_feat, feature_cols

Overwriting src/xgboost_model/train_xgboost_model.py


# forecasting

this predicts future unseen values (not just test data)

!!!! will need to debug this to align the dates with what sarimax outputs

In [11]:
%%writefile src/xgboost_model/xgboost_forecaster.py

"""
contains a class for forecasting the future value from the trained XGBoost model.
"""

import polars as pl
import xgboost as xgb
from datetime import timedelta

from src.load_data import load_stocks
from src.xgboost_model.xgboost_etl import *


class XGBStockForecaster:
    """
    use the trained XGBoost model to make a forecast over a specified interval.
    """
    def __init__(
        self, model: xgb.XGBRegressor, feature_cols: list, label: str
    ):
        self.model = model
        self.feature_cols = feature_cols
        self.label = label
    
    def _predict_from_features(self, df_feat: pl.DataFrame) -> float:
        """make a prediction based on the passed in features.

        Args:
            df_feat (pl.DataFrame): features table on which the model was
                trained.

        Returns:
            float: single predicted value.
        """
        row_pd = df_feat.select(self.feature_cols).tail(1).to_pandas()
        preds = self.model.predict(row_pd)
        return float(preds[0])
    
    def forecast_horizon(self, df_raw: pl.DataFrame, days: int) -> pl.DataFrame:
        """run a forecast on the full horizon indciated by `days`.

        Args:
            df_raw (pl.DataFrame): raw `yfinance` stock dataframe.
            days (int): number of days to forecast for.

        Returns:
            pl.DataFrame: table with a date and a predicted value column.
        """
        df_current = df_raw.clone()

        forecast_dates = []
        forecast_values = []

        for _ in range(days):
            df_feat = prep_data_frame(df_current)
            pred = self._predict_from_features(df_feat)
            last_date = df_current["date"][-1]
            next_date = last_date + timedelta(days=1)

            while next_date.weekday() >= 5:
                next_date = next_date + timedelta(days=1)
            
            forecast_dates.append(next_date)
            forecast_values.append(pred)

            last_row = df_current.tail(1)

            date_dtype = df_current.schema["date"]

            new_row = last_row.with_columns(
                pl.lit(next_date).cast(date_dtype).alias("date"),
                pl.lit(pred).alias(f"{self.label}")
            )

            df_current = df_current.vstack(new_row)
        
        return pl.DataFrame(
            {
                "date": forecast_dates,
                f"pred_{self.label}": forecast_values
            }
        )

Overwriting src/xgboost_model/xgboost_forecaster.py


# full modeling pipeline

raw data -> ETL -> split -> train a model -> evaluate model training -> forecast

In [12]:
%%writefile src/xgboost_model/xgboost_pipeline.py

"""
full pipeline for loading, transforming, training, and forecasting data for the
XGBoost model.
"""

import polars as pl
from datetime import datetime
from typing import Tuple

from src.load_data import load_stocks
from src.xgboost_model.train_xgboost_model import train_xgb_model
from src.xgboost_model.xgboost_forecaster import XGBStockForecaster

def train_and_forecast_xgb(
    ticker: str,
    start_date: str,
    end_date: str,
    cutoff: datetime,
    horizon_days: int,
    label: str = "close",
    n_estimators: int = 200,
    learning_rate: float = 0.05
) -> Tuple[pl.DataFrame, float]:
    """full pipeline for XGBoost model.

    Args:
        ticker (str): stock ticker to predict.
        start_date (str): when to start the training data.
        end_date (str): last day of the training data.
        cutoff (datetime): datetime object for train-test split.
        horizon_days (int): how many days in the future to forecast.
        label (str, optional): which value to predict. defaults to "close".
        n_estimators (int, optional): XGBoost `n_estimators` hyperparameter.
            defaults to 200.
        learning_rate (float, optional): XGBoost `learning_rate` hyperparameter.
            defaults to 0.05.

    Returns:
        Tuple[pl.DataFrame, float]: dataframe of predicted values per date and
            the RMSE from training.
    """
    stocks = [ticker]

    model, rmse, df_feat, feature_cols = train_xgb_model(
        stocks=stocks,
        start_date=start_date,
        end_date=end_date,
        cutoff=cutoff,
        label=label,
        n_estimators=n_estimators,
        learning_rate=learning_rate
    )

    df_raw = load_stocks(stocks, start_date, end_date)

    forecaster = XGBStockForecaster(model, feature_cols, label=label)
    forecasts_df = forecaster.forecast_horizon(df_raw, days=horizon_days)

    return forecasts_df, rmse

Overwriting src/xgboost_model/xgboost_pipeline.py


In [13]:
# this is fucking ugly

# refactor it once done with prophet

# i think this is where a class would come in handy

In [14]:
%%writefile src/sarimax_model/train_predict_sarimax_model.py

"""
full pipeline for loading the data, training, and forecasting with SARIMAX.
"""

import polars as pl
import pandas as pd
import pmdarima as pm 
from sklearn.metrics import root_mean_squared_error
from datetime import datetime, timedelta
from typing import Tuple

from src.load_data import load_stocks
from src.sarimax_model.sarimax_etl import *
from src.model_preprocess import split_ar_on_cutoff
from src.utils import build_forecast_dates


def fit_sarimax(
    ticker: str,
    start_date: str,
    end_date: str,
    label: str,
    cutoff: datetime,
    horizon_days: int,
    eval_mode: bool = True
):
    """fit the actual SARIMAX model.

    has two modes: eval and forecast. eval mode uses the train-eval split to 
    gauge how accurate the forecasts are (using RMSE). forecast mode uses the
    full dataset to train and make a forecast.

    Args:
        ticker (str): stock ticker to predict.
        start_date (str): when to start the training data.
        end_date (str): last day of the training data.
        label (str): which value to predict. defaults to "close".
        cutoff (datetime): datetime object for train-test split.
        horizon_days (int): how many days in the future to forecast.
        eval_mode (bool, optional): whether to run the model as an evaluation of
            performance or as a full forecast. defaults to True (i.e., evaluate 
            the model performance).

    Returns:
        _type_: output depends on `eval_mode`. returns a float (RMSE) if 
            `eval_mode` is True, or a table (date, predicted value) if 
            `eval_mode` is False.
    """
    stock = [ticker]

    df_raw = load_stocks(stock, start_date, end_date)
    df_idx = build_df_with_indices(df_raw, label, start_date, end_date)

    feats = [
        "date",
        "dow",
        "month",
        "mon_or_fri",
        "volume",
        "SPY_close",
        "SPY_volume",
        "QQQ_close",
        "QQQ_volume",
        "IWM_close",
        "IWM_volume",
        "VXX_close",
        "VXX_volume",
        "UUP_close",
        "UUP_volume",
        "HYG_close",
        "HYG_volume",
        "LQD_close",
        "LQD_volume"
    ]

    exog_cols = [feat for feat in feats if feat != "date"]

    _sarima_hyperparams = {
        "start_p": 1,
        "start_q": 1,
        "test": "adf",
        "max_p": 3,
        "max_q": 3,
        "m": 5,
        "start_P": 0,
        "seasonal": True,
        "d": None,
        "D": None,
        "trace": False,
        "error_action": "ignore",
        "suppress_warnings": True,
        "stepwise": True
    }
    
    if eval_mode:
        df_train, df_eval = split_ar_on_cutoff(df_idx, cutoff, "close", feats)

        sarimax_model = pm.auto_arima(
            df_train[[label]],
            exogenous=df_train[exog_cols],
            **_sarima_hyperparams
        )

        fitted, confint = sarimax_model.predict(
            n_periods=len(df_eval),
            return_conf_int=True,
            exogenous=df_eval[exog_cols]
        )

        fitted = pd.DataFrame(fitted, columns=["pred"]).reset_index(drop=True)
        df_eval["pred"] = fitted["pred"]
        rmse = root_mean_squared_error(df_eval[["close"]], df_eval[["pred"]])
        
        return rmse
    else:
        df = df_idx.to_pandas()

        sarimax_model = pm.auto_arima(
            df[[label]],
            exogenous=df[exog_cols],
            **_sarima_hyperparams
        )

        fitted, confint = sarimax_model.predict(
            n_periods=horizon_days,
            return_conf_int=True,
            exogenous=df[exog_cols]
        )
        fitted = pd.DataFrame(fitted, columns=[f"pred_{label}"]).reset_index(
            drop=True
        )
        ci_series = pd.DataFrame(
            confint, columns=["lower_bound", "upper_bound"]
        )
        
        df_out = build_forecast_dates(
            end_date, horizon_days, skip_weekends=True
        )
        
        df_out[f"pred_{label}"] = fitted[f"pred_{label}"]
        df_out["lower_bound"] = ci_series["lower_bound"]
        df_out["upper_bound"] = ci_series["upper_bound"]

        return df_out.sort_values(by="date", ascending=True)

def sarimax_wrapper(
    ticker: str,
    start_date: str,
    end_date: str,
    label: str,
    cutoff: datetime,
    horizon_days: int
) -> Tuple[pl.DataFrame, float]:
    """wrapper to run both versions of `fit_sarimax`.

    gets training results (`eval_mode == True`) and forecast results (`eval_mode
    == False`).

    Args:
        ticker (str): stock ticker to predict
        start_date (str): when to start the training data.
        end_date (str): last day of the training data.
        label (str): which value to predict. defaults to "close".
        cutoff (datetime): datetime object for train-test split.
        horizon_days (int): how many days in the future to forecast.

    Returns:
        Tuple[pl.DataFrame, float]: table of predictions (date, predicted value)
            and the RMSE from training.
    """
    rmse = fit_sarimax(
        ticker,
        start_date,
        end_date,
        label,
        cutoff,
        horizon_days,
        eval_mode=True
    )

    forecasts = fit_sarimax(
        ticker,
        start_date,
        end_date,
        label,
        cutoff,
        horizon_days,
        eval_mode=False
    )

    return pl.from_pandas(forecasts), rmse

Overwriting src/sarimax_model/train_predict_sarimax_model.py


In [15]:
## file for prophet pipeline

In [16]:
%%writefile src/utils.py

"""
utility functions
    build_forecast_dates
"""

from datetime import timedelta
import pandas as pd

def build_forecast_dates(
    last_date,
    horizon_days: int,
    skip_weekends: bool = True
) -> pd.DataFrame:
    """create a dataframe of dates based on `horizon_days`.

    Args:
        last_date (_type_): either a string date or datetime object.
        horizon_days (int): number of days (rows) to create dataframe of.
        skip_weekends (bool, optional): should weekends be skipped. defaults to
            True.

    Returns:
        pd.DataFrame: pandas dataframe of dates.
    """
    if not isinstance(last_date, (pd.Timestamp, )):
        last_date = pd.Timestamp(last_date)
    
    forecast_dates = []
    current_date = last_date 

    for _ in range(horizon_days):
        current_date = current_date + timedelta(days=1)

        if skip_weekends:
            while current_date.weekday() >= 5:
                current_date = current_date + timedelta(days=1)
        
        forecast_dates.append(current_date)
    
    return pd.DataFrame({"date": forecast_dates})

Overwriting src/utils.py


In [82]:
# %%writefile workflows/train_and_forecast_model.py

"""
script for getting all model results.
"""

from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
from timeit import default_timer as timer

import warnings
warnings.filterwarnings("ignore")

from src.xgboost_model.xgboost_pipeline import train_and_forecast_xgb
from src.sarimax_model.train_predict_sarimax_model import *

end_date = datetime.today().strftime("%Y-%m-%d")
start_date = (datetime.today() - relativedelta(years=10)).strftime("%Y-%m-%d")
cutoff = (datetime.today() - relativedelta(months=2))

ticker = "MSFT"
horizon = 10

print(f"\n\nforecasting '{ticker}' prices over the next {horizon} days")
print(
    f"training models from {start_date} to {end_date}, splitting on {cutoff.strftime("%Y-%m-%d")}\n\n"
)

timer_xgb_start = timer()

xgb_forecasts, xgb_rmse = train_and_forecast_xgb(
    ticker=ticker,
    start_date=start_date,
    end_date=end_date,
    cutoff=cutoff,
    horizon_days=horizon,
    label="close",
    n_estimators=200,
    learning_rate=0.05
)
timer_xgb_end = timer() - timer_xgb_start

print(f"\nXGBoost test RMSE on holdout: {xgb_rmse:.4f}\n")
print(f"XGBoost run duration: {timer_xgb_end:.5f} seconds\n\n")
print(xgb_forecasts)
print("\n\n")

timer_smax_start = timer()

smax_forecasts, smax_rmse = sarimax_wrapper(
    ticker=ticker,
    start_date=start_date,
    end_date=end_date,
    label="close",
    cutoff=cutoff,
    horizon_days=horizon,    
)
timer_smax_end = timer() - timer_smax_start

print(f"\nSARIMAX test RMSE on holdout: {smax_rmse:.4f}\n")
print(f"SARIMAX run duration: {timer_smax_end:.5f} seconds\n\n")
print(smax_forecasts)
print("\n\n")

[*********************100%***********************]  1 of 1 completed



forecasting 'MSFT' prices over the next 10 days
training models from 2015-11-24 to 2025-11-24, splitting on 2025-09-24





[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



XGBoost test RMSE on holdout: 9.7511

XGBoost run duration: 0.73418 seconds


shape: (10, 2)
┌─────────────────────┬────────────┐
│ date                ┆ pred_close │
│ ---                 ┆ ---        │
│ datetime[μs]        ┆ f64        │
╞═════════════════════╪════════════╡
│ 2025-11-24 00:00:00 ┆ 490.631134 │
│ 2025-11-25 00:00:00 ┆ 494.594849 │
│ 2025-11-26 00:00:00 ┆ 484.644287 │
│ 2025-11-27 00:00:00 ┆ 486.798279 │
│ 2025-11-28 00:00:00 ┆ 492.671234 │
│ 2025-12-01 00:00:00 ┆ 493.571136 │
│ 2025-12-02 00:00:00 ┆ 482.166687 │
│ 2025-12-03 00:00:00 ┆ 482.426514 │
│ 2025-12-04 00:00:00 ┆ 495.229767 │
│ 2025-12-05 00:00:00 ┆ 493.825165 │
└─────────────────────┴────────────┘





[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********


SARIMAX test RMSE on holdout: 13.5721

SARIMAX run duration: 4.91360 seconds


shape: (10, 4)
┌─────────────────────┬────────────┬─────────────┬─────────────┐
│ date                ┆ pred_close ┆ lower_bound ┆ upper_bound │
│ ---                 ┆ ---        ┆ ---         ┆ ---         │
│ datetime[ns]        ┆ f64        ┆ f64         ┆ f64         │
╞═════════════════════╪════════════╪═════════════╪═════════════╡
│ 2025-11-25 00:00:00 ┆ 479.239865 ┆ 470.413078  ┆ 488.066651  │
│ 2025-11-26 00:00:00 ┆ 479.440158 ┆ 467.394402  ┆ 491.485913  │
│ 2025-11-27 00:00:00 ┆ 479.640451 ┆ 465.070335  ┆ 494.210567  │
│ 2025-11-28 00:00:00 ┆ 479.840744 ┆ 463.123198  ┆ 496.558291  │
│ 2025-12-01 00:00:00 ┆ 480.041037 ┆ 461.422111  ┆ 498.659964  │
│ 2025-12-02 00:00:00 ┆ 480.241331 ┆ 459.897966  ┆ 500.584695  │
│ 2025-12-03 00:00:00 ┆ 480.441624 ┆ 458.508987  ┆ 502.37426   │
│ 2025-12-04 00:00:00 ┆ 480.641917 ┆ 457.227635  ┆ 504.056199  │
│ 2025-12-05 00:00:00 ┆ 480.84221  ┆ 456.034617  ┆ 505.64980

In [ ]:
import polars as pl
import pandas as pd  
from datetime import datetime, date, timedelta 
from prophet import Prophet
from sklearn.metrics import root_mean_squared_error
from typing import Tuple

import warnings
warnings.filterwarnings("ignore")

from src.load_data import load_stocks
from src.utils import build_forecast_dates
from src.prophet_model.prophet_etl import build_prophet_df
from src.model_preprocess import split_prophet_df

stk = ["AAPL"]
start_date = "2015-09-01"
cutoff = datetime(2025, 7, 1, 0, 0, 0)
end_date = "2025-09-01" 
label = "close"
idxs = ["VXX", "QQQ", "SPY", "IWM"]

df_prophet = build_prophet_df(
    stk,
    start_date,
    end_date,
    label,
    idxs
)

df_train, df_eval = split_prophet_df(df_prophet, cutoff)


### basic prophet model
reg_cols = [c for c in df_prophet.columns if c not in ["ds", "y"]]

pr_mod = Prophet(interval_width=0.95)
for col in reg_cols:
    pr_mod.add_regressor(col)

pr_mod.fit(df_train[["ds", "y"] + reg_cols])

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
18:12:40 - cmdstanpy - INFO - Chain [1] start processing
18:12:41 - cmdstanpy - INFO - Chain [1] done processing


In [22]:
df_eval_future = df_eval[["ds"] + reg_cols].copy()
forecast_eval = pr_mod.predict(df_eval_future)
df_forecast = forecast_eval[["ds", "yhat", "yhat_lower", "yhat_upper"]]
df_eval = df_eval[["ds", "y"]]

df_eval = df_eval.merge(df_forecast, on="ds", how="inner")
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(df_eval["y"], df_eval["yhat"])
print(rmse)
df_eval.head(20)

1.2697156098180713


,ds,y,yhat,yhat_lower,yhat_upper
0,2025-07-01,207.383377,207.638400,205.421282,209.592034
1,2025-07-02,211.993668,211.015255,209.092103,213.053864
2,2025-07-03,213.101334,213.439040,211.397638,215.579193
3,2025-07-07,209.508896,211.742641,209.672739,213.982967
4,2025-07-08,209.568771,210.253007,208.263915,212.345744
5,2025-07-09,210.696396,210.713104,208.725188,212.798713
6,2025-07-10,211.963745,212.192436,210.027229,214.260495
7,2025-07-11,210.716354,212.066558,210.080380,214.190123
8,2025-07-14,208.181686,209.904291,207.697565,211.836267
9,2025-07-15,208.670670,209.293998,207.145180,211.429156


In [39]:
from src.utils import build_forecast_dates

horizon_days = 10

last_date = df_prophet["ds"].max()
future_dates = build_forecast_dates(
    last_date=last_date,
    horizon_days=horizon_days,
    skip_weekends=True
)

last_row = df_prophet.sort_values("ds").iloc[-1]

date_list = future_dates["date"].tolist()

rows = []
for d in date_list:
    row = {"ds": d}
    for col in reg_cols:
        row[col] = last_row[col]
    rows.append(row)

future_forward = pd.DataFrame(rows)

forecast_forward = pr_mod.predict(future_forward)

df_out = forecast_forward[["ds", "yhat", "yhat_lower", "yhat_upper"]]
df_out

,ds,yhat,yhat_lower,yhat_upper
0,2025-08-29,232.101015,230.024187,234.107005
1,2025-09-01,232.025106,229.969630,234.048896
2,2025-09-02,232.037844,230.013243,234.100414
3,2025-09-03,231.950433,229.750843,233.812764
4,2025-09-04,231.998026,229.905396,234.033962
5,2025-09-05,232.109668,230.141054,234.233688
6,2025-09-08,232.087508,230.076090,234.283908
7,2025-09-09,232.118135,229.948202,234.284781
8,2025-09-10,232.047960,230.121371,234.008423
9,2025-09-11,232.111764,230.039395,234.215067


In [ ]:
%%writefile src/prophet_model/prophet_model_pipeline.py

import pandas as pd 
import polars as pl 
from prophet import Prophet 
from datetime import datetime
from typing import Tuple
from sklearn.metrics import root_mean_squared_error

import warnings 
warnings.filterwarnings("ignore")

from src.prophet_model.prophet_etl import build_prophet_df
from src.utils import build_forecast_dates


class ProphetForecaster:
    """wrapper class to train and forecast Prophet model."""

    def __init__(self, base_params: dict | None = None):
        self.base_params = base_params or {"interval_width": 0.95}
        self.model = None
        self.reg_cols = None 
    
    def train_model(
        self,
        stock: list,
        start_date: str,
        end_date: str,
        cutoff: datetime,
        label: str = "close",
        indexes: list[str] | None = None
    ) -> Tuple[float, pd.DataFrame]:
        """build data, train and evaluate Prophet model.

        data includes the stock label (defaults to close), index fund values,
        returns, RSI (7, 14, 21 days), and lag features of label (1, 7, 30 days)
        to predict future label values.

        Args:
            stock (list): stock to be predicted.
            start_date (str): start of historical price data.
            end_date (str): end of historical price data.
            cutoff (datetime): date to split for train/eval.
            label (str, optional): value to predict. defaults to "close".
            indexes (list[str] | None, optional): specific index funds to add to
                feature space. defaults to None.

        Returns:
            Tuple[float, pd.DataFrame]: RMSE value for evaluation and eval
                dataframe.
        """
        df = build_prophet_df(stock, start_date, end_date, label, indexes)
        df_train, df_eval = split_prophet_df(df, cutoff)

        self.reg_cols = [c for c in df.columns if c not in ["ds", "y"]]

        pr_model = Prophet(**self.base_params)
        for col in self.reg_cols:
            pr_model.add_regressor(col)

        pr_model.fit(df_train[["ds", "y"] + self.reg_cols])
        self.model = pr_model

        df_eval_future = df_eval[["ds"] + self.reg_cols].copy()
        forecast_eval = self.model.predict(df_eval_future)

        df_forecast = forecast_eval[["ds", "yhat", "yhat_lower", "yhat_upper"]]
        df_eval_out = df_eval[["ds", "y"]]
        df_eval_out = df_eval_out.merge(df_forecast, on="ds", how="inner")

        rmse = root_mean_squared_error(df_eval["y"], df_eval_out["yhat"])
        
        return rmse, df_eval_out
    
    def _build_future_regressors(
            self, df_full: pd.DataFrame, future_dates: pd.DataFrame
        ) -> pd.DataFrame:
        """stand-in function to make a dataframe for future predictions.

        will need to be added to if this were to ever go live for the sake of
        adding future regressors (and not just dates).

        Args:
            df_full (pd.DataFrame): full features dataframe.
            future_dates (pd.DataFrame): dataframe of days in the future.

        Returns:
            pd.DataFrame: single row containing date and regressors for making
                the forecast.
        """
        last_row = df_full.sort_values("ds").iloc[-1]
        date_list = future_dates["date"].tolist()
        rows = []
        for d in date_list:
            row = {"ds": d}
            for col in reg_cols:
                row[col] = last_row[col]
            rows.append(row)
        
        return pd.DataFrame(rows)
    
    def make_prediction(
        self,
        df_full: pd.DataFrame,
        horizon_days: int,
        label: str = "close"
    ) -> pl.DataFrame:
        """generate predictions `horizon_days` in the future.

        returns a polars dataframe for in-line displays.

        Args:
            df_full (pd.DataFrame): full features dataframe.
            horizon_days (int): number of days to forecast into the future.
            label (str, optional): specific value to forecast. defaults to 
            "close".

        Returns:
            pl.DataFrame: polars dataframe containing future dates, predicted
                values, and upper/lower bound CIs (95%).
        """
        assert self.model is not None 

        last_date = df_full["ds"].max()

        future_dates = build_forecast_dates(
            last_date,
            horizon_days=horizon_days,
            skip_weekends=True 
        )

        future_regs = self._build_future_regressors(df_full, future_dates)

        future_df = future_regs[["ds"] + self.reg_cols]

        forecast = self.model.predict(future_df)
        out = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
        df_out = pl.from_pandas(out)

        return df_out.select(
            pl.col("ds").alias("date"),
            pl.col("yhat").alias(f"pred_{label}"),
            pl.col("yhat_lower").alias("lower_bound"),
            pl.col("yhat_upper").alias("upper_bound")
        )

Writing src/prophet_model/prophet_model_pipeline.py


In [1]:
import pandas as pd 
import polars as pl 
from datetime import datetime 

import warnings
warnings.filterwarnings("ignore")

from src.prophet_model.prophet_model_pipeline import ProphetForecaster
from src.utils import build_forecast_dates
from src.prophet_model.prophet_etl import build_prophet_df
from src.model_preprocess import split_prophet_df

stk = ["AAPL"]
start_date = "2015-09-01"
cutoff = datetime(2025, 7, 1, 0, 0, 0)
end_date = "2025-09-01" 
label = "close"
idxs = ["VXX", "QQQ", "SPY", "IWM"]

forecaster = ProphetForecaster()

rmse, eval_df = forecaster.train_model(
    stk, start_date, end_date, cutoff, label, idxs
)

df_full = build_prophet_df(stk, start_date, end_date, label, idxs)
print(rmse)

df_pred = forecaster.make_prediction(df_full, horizon_days=10)

print(df_pred)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


NameError: name 'split_prophet_df' is not defined